# M5 — Suggested route

**Approach:** summary-only router with a review gate (contributor F). Word + character TF-IDF into class-balanced logistic regression, **chronological split**. Channel / plan / tenure are excluded (their link to the team drifts month to month); `operations` is merged into `support`. A top suggestion below the review threshold (0.5) is marked *needs human review*. Saves `models/m5_route.joblib`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "northstar_prep.py")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import numpy as np, pandas as pd
import northstar_prep as P

In [ ]:
df = P.load_clean(merge_team_aliases=True)
FEATS, TARGET, REVIEW = ["case_summary"], "assigned_team", 0.5
labels = sorted(df[TARGET].unique())
tr, va, te = P.time_split(df)
pipe = P.make_clf([P.t_word(), P.t_char()]).fit(tr[FEATS], tr[TARGET])
proba = pipe.predict_proba(te[FEATS]); pred = pipe.classes_[proba.argmax(1)]; conf = proba.max(1)
base = df[TARGET].value_counts(normalize=True).iloc[0]
metrics = P.evaluate(te[TARGET], pred, labels, baseline=base)
metrics["pct_needs_review"] = round(float((conf < REVIEW).mean()), 3)
mask = conf >= REVIEW
metrics["acc_when_confident"] = round(float(P.accuracy_score(te[TARGET][mask], pred[mask])), 3)
metrics

In [ ]:
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["predicted"], test["confidence"] = pred, conf.round(3)
full = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATS], df[TARGET])
P.save_bundle("m5_route", {
    "model": full, "target": TARGET, "features": FEATS, "classes": labels, "kind": "multiclass",
    "label": "Suggested route", "review_threshold": REVIEW, "metrics": metrics, "test": test,
    "report": P.class_report(te[TARGET], pred, labels), "confusion": P.conf_matrix(te[TARGET], pred, labels),
    "meta": {"contributor":"F", "approach":"summary-only router with review gate",
             "split":"time", "inputs":"summary only",
             "note":"operations merged into support; review gate at 0.5"}})